# PoBot — QLoRA Fine-Tuning (Colab)

Fine-tunes a small open model (default **Llama-3.2-1B-Instruct**) to answer
Hong Kong labour questions in PoBot's grounded, cited style, using the dataset
built by `build_dataset.py`.

**Runtime:** Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

**Honest framing:** this is *knowledge distillation* from the Groq 70B teacher
and a demonstration of the PEFT/LoRA skill + a cheap self-hostable model. A 1B
student won't beat the 70B on hard queries; its value is format discipline
(grounding, citations, refusal) and low-cost/offline deployment.

All hyperparameters live in `finetuning/config.yaml` (single source of truth).


## 1. Check the GPU


In [1]:
!nvidia-smi


Fri Jul 10 12:38:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install pinned dependencies


In [2]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes trl pyyaml


## 3. Get the repo (dataset + config)

Replace the URL with your repo. The dataset (`finetuning/data/*.jsonl`) and
`config.yaml` are versioned there, so this is reproducible.


In [3]:
# Option A: clone your repo
!git clone https://github.com/uzairahim/PoBot_RAG-Chatbot-for-Hong-Kong-Labour-Regulations.git pobot
%cd pobot/finetuning

# Option B (if not using git): upload train.jsonl, val.jsonl, config.yaml via the
# Files panel, then set the paths below accordingly.

fatal: destination path 'pobot' already exists and is not an empty directory.
/content/pobot/finetuning


## 4. Hugging Face auth

Llama-3.2 is a **gated** model — request access on its HF page, then add your
token to Colab **Secrets** (key icon, left sidebar) as `HF_TOKEN`.


In [4]:
# from google.colab import userdata
# from huggingface_hub import login
# login(userdata.get('HF_TOKEN'))

from huggingface_hub import login
import getpass
login(getpass.getpass("HF token: "))


## 5. Load config


In [5]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
BASE = cfg['base_model']
L, Q, T = cfg['lora'], cfg['quantization'], cfg['training']
print('base model:', BASE)
print('LoRA r/alpha:', L['r'], L['alpha'], '| epochs:', T['epochs'])


base model: meta-llama/Llama-3.2-1B-Instruct
LoRA r/alpha: 16 32 | epochs: 3


## 6. Load the dataset and render to text with the chat template


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ds = load_dataset('json', data_files={
    'train': cfg['dataset']['train_file'],
    'validation': cfg['dataset']['val_file'],
})

def to_text(ex):
    # Render our [{role,content}...] messages into the model's chat format.
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}

ds = ds.map(to_text, remove_columns=ds['train'].column_names)
print(ds)
print(ds['train'][0]['text'][:800])


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3
    })
})
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 Jul 2026

You are PoBot, an assistant that answers questions about Hong Kong labour and employment regulations for migrant workers (including foreign domestic helpers).

Rules:
- Answer ONLY using the numbered CONTEXT passages provided. Do not use outside knowledge.
- Cite the passages you rely on inline using their bracket numbers, e.g. [1], [2].
- If the context does not contain enough information to answer, say so plainly and suggest contacting the Hong Kong Labour Department. Do not guess.
- Reply in the SAME language as the user's question (e.g. English or Tagalog).
- Be clear, concise, and practical. Use plain language a worker can understand.
- End with a one-line remind


## 7. Load the base model in 4-bit (QLoRA) + attach LoRA


In [7]:
import os
# Reduce allocator fragmentation on the T4 (the "reserved but unallocated" bytes
# in an OOM). Must be set before the first CUDA allocation below.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=Q['load_in_4bit'],
    bnb_4bit_quant_type=Q['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=Q['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=getattr(torch, Q['bnb_4bit_compute_dtype']),
)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')

# QLoRA memory recipe (this is what lets training fit on a 15 GB T4):
# - gradient checkpointing: recompute layer activations during backward instead
#   of holding them all in VRAM -> the big saving.
# - prepare_model_for_kbit_training also casts layernorms to fp32 and calls
#   enable_input_require_grads() so gradients flow to the LoRA adapters through
#   the frozen, checkpointed 4-bit layers.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # incompatible with gradient checkpointing

lora = LoraConfig(
    r=L['r'], lora_alpha=L['alpha'], lora_dropout=L['dropout'],
    target_modules=L['target_modules'], task_type='CAUSAL_LM', bias='none',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()  # note the tiny % that is trainable


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## 8. Train — answer-only loss

Cross-entropy is computed **only on the answer tokens**: every prompt token
(the retrieved context + the question) is masked to `-100` so the model is
optimized on *how to answer* in PoBot’s cited style — not on echoing back the
context we already gave it. Uses a plain `transformers.Trainer` with manual
masking, which is version-proof across TRL releases.


In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# answer-only loss, done explicitly (works on ANY trl/transformers
# version). We tokenize each example and set label = -100 for every token up to
# and including the assistant header, so cross-entropy is computed ONLY on the
# answer tokens. (-100 is PyTorch's ignore_index.) The model shifts labels by
# one internally, so labels stay aligned with input_ids here.
RESP = tok.encode("<|start_header_id|>assistant<|end_header_id|>", add_special_tokens=False)

def tokenize_and_mask(ex):
    enc = tok(ex["text"], truncation=True, max_length=T["max_seq_length"])
    ids = enc["input_ids"]
    labels = ids.copy()
    start = 0
    for i in range(len(ids) - len(RESP) + 1):        # locate the assistant header
        if ids[i:i + len(RESP)] == RESP:
            start = i + len(RESP)
            break
    for i in range(start):                            # mask prompt tokens -> ignored
        labels[i] = -100
    enc["labels"] = labels
    return enc

tokds = ds.map(tokenize_and_mask, remove_columns=ds["train"].column_names)

# sanity check: confirm the prompt really is masked on example 0
_lab = tokds["train"][0]["labels"]
_m = sum(1 for x in _lab if x == -100)
print(f"example 0: {_m}/{len(_lab)} tokens masked (prompt, ignored), "
      f"{len(_lab) - _m} graded (answer)")

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=T["epochs"],
    learning_rate=T["learning_rate"],
    per_device_train_batch_size=T["per_device_batch_size"],
    gradient_accumulation_steps=T["gradient_accumulation_steps"],
    warmup_ratio=T["warmup_ratio"],
    logging_steps=T["logging_steps"],
    save_steps=T["save_steps"],
    eval_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",  # paged optimizer: avoids VRAM spikes from Adam state
    report_to="none",
)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
trainer = Trainer(
    model=model, args=args,
    train_dataset=tokds["train"], eval_dataset=tokds["validation"],
    data_collator=collator,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


example 0: 894/1000 tokens masked (prompt, ignored), 106 graded (answer)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,0.596769
2,No log,0.563269
3,1.833673,0.546476


TrainOutput(global_step=12, training_loss=1.700392723083496, metrics={'train_runtime': 447.2382, 'train_samples_per_second': 0.215, 'train_steps_per_second': 0.027, 'total_flos': 559807188099072.0, 'train_loss': 1.700392723083496, 'epoch': 3.0})

## 9. Eyeball base vs fine-tuned

Qualitative check on a couple of validation prompts — look for grounding,
citations, and refusal behavior.


In [9]:
# Base vs fine-tuned on the same validation prompt.
def generate(text):
    prompt = text.split('Answer:')[0] + 'Answer:'   # cut before the target answer
    ins = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True).strip()

sample = ds['validation'][0]['text']

# disable_adapter() turns the LoRA OFF -> we get the frozen BASE model's answer
# from the SAME loaded weights (no second copy in memory).
with model.disable_adapter():
    print("=== BASE (Llama-3.2-1B, no adapter) ===")
    print(generate(sample))

print("\n=== FINE-TUNED (base + LoRA adapter) ===")
print(generate(sample))

=== BASE (Llama-3.2-1B, no adapter) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


According to the Employment Ordinance, if an employer needs you to work on your scheduled rest day due to an emergency, they must substitute another rest day with your original rest day. However, they must do this within 30 days of the original rest day.

=== FINE-TUNED (base + LoRA adapter) ===
Your employer must substitute some other rest day with the consent of the employee, in which case it must be within the same month before the original rest day or within 30 days after it. Compulsory Work on Rest days An employer must not compel an employee to work on a rest day except in the event of a breakdown of machinery or plant or in any other unforeseen emergency. For any rest day on which the employee is required to work, the employer should substitute some other rest day within 30 days after the original rest day. The employer should notify the employee of the arrangement within 48 hours after the employee is required to work.


## 10. Save the adapter

Saves the LoRA adapter (~10-50 MB), zips it, and downloads it. Commit it to
`finetuning/adapter/` in the repo, then run locally with `LLM_PROVIDER=hf_local`.


In [10]:
adapter_dir = cfg['output']['adapter_dir']
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

import shutil
from google.colab import files
shutil.make_archive('pobot_adapter', 'zip', adapter_dir)
files.download('pobot_adapter.zip')

# Optional: push to the Hugging Face Hub instead of downloading
# model.push_to_hub('<you>/pobot-hk-labour-lora')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

1. Unzip the adapter into `finetuning/adapter/` and commit it.
2. Locally: `pip install -r requirements-finetune.txt`.
3. In `.env`: `LLM_PROVIDER=hf_local` and `HF_ADAPTER_DIR=finetuning/adapter`.
4. `python chatbot.py` — the fine-tuned model now answers through the RAG pipeline.
